In [106]:
import sys
from pyspark import SparkConf, SparkContext
from math import sqrt

In [107]:
import os

# Create the ml-100k directory if it doesn't exist
if not os.path.exists('ml-100k'):
    os.makedirs('ml-100k')
    print("Directory 'ml-100k' created.")
else:
    print("Directory 'ml-100k' already exists.")

!wget -O ml-100k/u.data https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/ml-100k/u.data
!wget -O ml-100k/u.item https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/ml-100k/u.item

Directory 'ml-100k' already exists.
--2026-02-21 08:45:40--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/ml-100k/u.data
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1979173 (1.9M) [text/plain]
Saving to: ‘ml-100k/u.data’

ml-100k/u.data      100%[===================>]   1.89M  9.16MB/s    in 0.2s    

2026-02-21 08:45:41 (9.16 MB/s) - ‘ml-100k/u.data’ saved [1979173/1979173]

--2026-02-21 08:45:41--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/ml-100k/u.item
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting res

In [108]:
def loadMovieNames():
    movieNames = {}
    with open("ml-100k/u.item", encoding='iso-8859-1') as f:
        for line in f:
            fields = line.strip().split('|')
            if len(fields) >= 2 and fields[0].isdigit():
                movieNames[int(fields[0])] = fields[1]
    return movieNames

In [109]:
def makePairs(user_ratings):
    user, ratings = user_ratings
    (movie1, rating1) = ratings[0]
    (movie2, rating2) = ratings[1]
    return ((movie1, movie2), (rating1, rating2))

In [110]:
def computeCosineSimilarity(ratingPairs):
    numPairs = 0
    sum_xx = sum_yy = sum_xy = 0

    for ratingX, ratingY in ratingPairs:
        sum_xx += ratingX * ratingX
        sum_yy += ratingY * ratingY
        sum_xy += ratingX * ratingY
        numPairs += 1

    numerator = sum_xy
    denominator = sqrt(sum_xx) * sqrt(sum_yy)

    score = 0.0
    if denominator:
        score = numerator / float(denominator)
    return (score, numPairs)

In [111]:
conf = SparkConf().setMaster("local[*]").setAppName("MovieSimilarities")
sc = SparkContext.getOrCreate(conf=conf)

In [112]:
print("\nLoading movie names...")
nameDict = sc.broadcast(loadMovieNames()).value

data = sc.textFile("ml-100k/u.data")

ratings = data.map(lambda l: l.split()).map(lambda l: (int(l[0]), (int(l[1]), float(l[2]))))
ratings = ratings.filter(lambda x: x[1][1] > 2.0)

joinedRatings = ratings.join(ratings)

filtered = joinedRatings.filter(lambda x: x[1][0][0] < x[1][1][0])

moviePairs = filtered.map(makePairs)

moviePairRatings = moviePairs.groupByKey()

moviePairSimilarities = moviePairRatings.mapValues(computeCosineSimilarity).cache()

scoreThreshold = 0.95
coOccurenceThreshold = 35
movieIDs = [50, 67, 73]

for movieID in movieIDs:
  filteredResults = moviePairSimilarities.filter(
      lambda x: ((x[0][0] == movieID) or (x[0][1] == movieID))
                and (x[1][0] > scoreThreshold)
                and (x[1][1] > coOccurenceThreshold)
  )
  results = (filteredResults.map(lambda x: (x[1], x[0])).sortByKey(ascending=False).take(20))

  print("\n=== Top similar movies for " + nameDict[movieID] + " (" + str(movieID) + ") ====")
  for result in results:
      (sim, pair) = result

      similarMovieID = pair[0]
      if (pair[0] == movieID):
          similarMovieID = pair[1]

      print(nameDict[similarMovieID] + "\tscore: " + str(round(sim[0], 2)) + "\tstrength: " + str(sim[1]))

sc.stop()


Loading movie names...

=== Top similar movies for Star Wars (1977) (50) ====
Empire Strikes Back, The (1980)	score: 0.99	strength: 327
Return of the Jedi (1983)	score: 0.99	strength: 450
Wallace & Gromit: The Best of Aardman Animation (1996)	score: 0.99	strength: 56
20,000 Leagues Under the Sea (1954)	score: 0.99	strength: 62
Wrong Trousers, The (1993)	score: 0.99	strength: 97
Raiders of the Lost Ark (1981)	score: 0.99	strength: 358
Hamlet (1996)	score: 0.98	strength: 73
Close Shave, A (1995)	score: 0.98	strength: 88
Sting, The (1973)	score: 0.98	strength: 191
Mrs. Brown (Her Majesty, Mrs. Brown) (1997)	score: 0.98	strength: 42
Secret of Roan Inish, The (1994)	score: 0.98	strength: 54
Philadelphia Story, The (1940)	score: 0.98	strength: 82
Roman Holiday (1953)	score: 0.98	strength: 47
Terminator 2: Judgment Day (1991)	score: 0.98	strength: 236
North by Northwest (1959)	score: 0.98	strength: 147
Tommy Boy (1995)	score: 0.98	strength: 36
Tomorrow Never Dies (1997)	score: 0.98	strength: